# Stacking — Tầng 2 (tự so sánh các bộ OOF, tối ưu macro-F1)

**Mục đích:** thử meta-learner trên **nhiều bộ xác suất OOF** (mặc định, có class weight `balanced`, `sqrt`), chọn cấu hình tốt nhất theo **macro-F1 trên dev**, rồi xuất submission.

**Các bộ OOF được tự phát hiện:**

| Variant | Thư mục | Nguồn dev/test |
|---|---|---|
| `default` | `artifacts/probs/oof/` | `artifacts/probs/no-oof/` (model fit toàn train) |
| `balanced` | `artifacts/probs/oof_balanced/` | cùng thư mục (trung bình 5 fold) |
| `sqrt` | `artifacts/probs/oof_sqrt/` | cùng thư mục (trung bình 5 fold) |

Variant nào chưa có file thì tự bỏ qua. Cần ít nhất 2 model mỗi variant.

**PhoBERT:** nếu `artifacts/probs/oof_phobert/` có đủ 3 file (`phobert_oof/dev/test.npy`), notebook tự thêm `phobert` làm base model thứ 6 cho mọi variant.

**Metric mục tiêu: macro-F1.**

**Lưu ý:** `class_weight="balanced"` cho meta đã thử và **làm giảm** macro-F1 (0.72 vs 0.756) vì prior tuning đã xử lý cân bằng lớp; notebook này chỉ dùng `class_weight=None` cho meta.

In [1]:
# ============================================================
# 1. IMPORT & CẤU HÌNH
# ============================================================
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
SEED = 42

BASE_MODELS = ["lightgbm", "linsvc", "logreg", "complement_nb", "sgd"]
USE_METADATA = True
USE_LOG_PROBS = True
META_COLS = ["rating", "log_num_sold", "log_num_review"]
C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 10.0]
CLASS_WEIGHTS = [None]   # balanced ở meta đã thử và làm giảm macro-F1


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PROBS_ROOT = ARTIFACT_DIR / "probs"
DENSE_DIR = ARTIFACT_DIR / "dense"
META_MODEL_DIR = ARTIFACT_DIR / "models" / "meta"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
META_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: D:\DS102_NguyenKhacHuy_24520679\Final project


## 2. Phát hiện các bộ OOF

Tự quét `artifacts/probs/oof/` (default) và các thư mục `oof_*` (ví dụ `oof_balanced`, `oof_sqrt`). Một variant chỉ được dùng nếu mỗi model có đủ 3 file `_oof`, `_dev`, `_test`.

In [2]:
# ============================================================
# 2. PHÁT HIỆN VARIANT
# ============================================================
def variant_specs():
    # default: OOF lấy từ probs/oof/, dev/test lấy từ probs/no-oof/
    # (nếu chưa tách thư mục thì dùng file lẻ ngay trong probs/)
    dev_dir = PROBS_ROOT / "no-oof" if (PROBS_ROOT / "no-oof").exists() else PROBS_ROOT
    specs = {
        "default": {"oof": PROBS_ROOT / "oof", "dev": dev_dir, "test": dev_dir}
    }
    for d in sorted(PROBS_ROOT.glob("oof_*")):
        if d.is_dir() and d.name != "oof_phobert":
            specs[d.name[len("oof_"):]] = {"oof": d, "dev": d, "test": d}
    return specs


def available_models(spec):
    ok = []
    for m in BASE_MODELS:
        if all((spec[s] / f"{m}_{s}.npy").exists() for s in ["oof", "dev", "test"]):
            ok.append(m)
    return ok


DATA = {}
for variant, spec in variant_specs().items():
    models = available_models(spec)
    if len(models) < 2:
        print(f"{variant:10s} bỏ qua (chỉ có {len(models)} model đủ file)")
        continue
    DATA[variant] = {
        "spec": spec,
        "models": models,
        "train": {m: np.load(spec["oof"] / f"{m}_oof.npy") for m in models},
        "dev": {m: np.load(spec["dev"] / f"{m}_dev.npy") for m in models},
        "test": {m: np.load(spec["test"] / f"{m}_test.npy") for m in models},
    }
    print(f"{variant:10s} models={models}")

if not DATA:
    raise FileNotFoundError(
        "Không có bộ OOF nào đủ file. Hãy chạy src/oof_probabilities.ipynb "
        "(và src/oof_class_weight.ipynb nếu muốn)."
    )

# --- tuỳ chọn: thêm PhoBERT vào mọi variant nếu đã có xác suất ---
EXTRA_MODEL = "phobert"
EXTRA_DIR = PROBS_ROOT / "oof_phobert"
extra_ok = all((EXTRA_DIR / f"{EXTRA_MODEL}_{s}.npy").exists() for s in ["oof", "dev", "test"])
if extra_ok:
    extra = {s: np.load(EXTRA_DIR / f"{EXTRA_MODEL}_{s}.npy") for s in ["oof", "dev", "test"]}
    for variant in DATA:
        if EXTRA_MODEL in DATA[variant]["models"]:
            continue
        DATA[variant]["models"].append(EXTRA_MODEL)
        DATA[variant]["train"][EXTRA_MODEL] = extra["oof"]
        DATA[variant]["dev"][EXTRA_MODEL] = extra["dev"]
        DATA[variant]["test"][EXTRA_MODEL] = extra["test"]
    print(f"Đã thêm {EXTRA_MODEL} vào các variant:", list(DATA))
else:
    print(f"Chưa có {EXTRA_DIR} đủ file — stacking không dùng {EXTRA_MODEL}.")

default    models=['lightgbm', 'linsvc', 'logreg', 'complement_nb', 'sgd']
balanced   models=['lightgbm', 'linsvc', 'logreg', 'complement_nb', 'sgd']
sqrt       models=['lightgbm', 'linsvc', 'logreg', 'complement_nb', 'sgd']
Đã thêm phobert vào các variant: ['default', 'balanced', 'sqrt']


In [3]:
# ============================================================
# 3. NHÃN & METADATA
# ============================================================
y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
y_dev = np.load(ARTIFACT_DIR / "labels" / "y_dev.npy")

pre = joblib.load(ARTIFACT_DIR / "models" / "preprocessors.joblib")
DENSE_NAMES = pre["dense_feature_names"]
META_IDX = [DENSE_NAMES.index(c) for c in META_COLS]

dense_train = np.load(DENSE_DIR / "dense_train.npy")[:, META_IDX]
dense_dev = np.load(DENSE_DIR / "dense_dev.npy")[:, META_IDX]
dense_test = np.load(DENSE_DIR / "dense_test.npy")[:, META_IDX]

meta_scaler = StandardScaler().fit(dense_train)
DENSE_S = {
    "train": meta_scaler.transform(dense_train),
    "dev": meta_scaler.transform(dense_dev),
    "test": meta_scaler.transform(dense_test),
}
print("Metadata:", META_COLS, "| shape:", DENSE_S["train"].shape)

Metadata: ['rating', 'log_num_sold', 'log_num_review'] | shape: (14306, 3)


## 4. Hàm dựng meta & tune prior

`log(p)` thay vì `p` để ổn định khi xác suất sát 0/1. Prior tune theo **macro-F1**.

In [4]:
# ============================================================
# 4. BUILD META + MACRO-F1 + PRIOR
# ============================================================
def _transform(p):
    return np.log(np.clip(p, 1e-6, 1.0)) if USE_LOG_PROBS else p


def build_meta(variant, split):
    d = DATA[variant]
    blocks = [_transform(d[split][m]) for m in d["models"]]
    if USE_METADATA:
        blocks.append(DENSE_S[split])
    return np.hstack(blocks)


def macro_f1(y_true, pred):
    return float(f1_score(y_true, pred, average="macro"))


PRIOR_GRID = [0.3, 0.4, 0.5, 0.6, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 2.5, 3.0]


def tune_prior(proba, y, passes=3):
    w = np.ones(4)
    best = macro_f1(y, (proba * w).argmax(axis=1))
    for _ in range(passes):
        improved = False
        for k in range(4):
            for g in PRIOR_GRID:
                w2 = w.copy()
                w2[k] = g
                score = macro_f1(y, (proba * w2).argmax(axis=1))
                if score > best + 1e-12:
                    best, w, improved = score, w2, True
        if not improved:
            break
    return w, best


print("Đã định nghĩa hàm.")

Đã định nghĩa hàm.


## 5. Grid search: variant × C

Với mỗi variant × `C`, fit meta trên OOF train, tune prior trên dev, ghi lại macro-F1 và accuracy.

In [5]:
# ============================================================
# 5. GRID SEARCH
# ============================================================
rows = []
for variant in DATA:
    Xtr = build_meta(variant, "train")
    Xdv = build_meta(variant, "dev")
    for C in C_GRID:
        for cw in CLASS_WEIGHTS:
            meta = LogisticRegression(
                C=C, class_weight=cw, max_iter=3000, solver="lbfgs", random_state=SEED
            )
            meta.fit(Xtr, y_train)
            p = meta.predict_proba(Xdv)
            pred0 = p.argmax(axis=1)
            w, best_f1 = tune_prior(p, y_dev)
            predw = (p * w).argmax(axis=1)
            rows.append({
                "variant": variant,
                "n_models": len(DATA[variant]["models"]),
                "C": C,
                "class_weight": str(cw),
                "mf1_no_prior": round(macro_f1(y_dev, pred0), 4),
                "mf1_prior": round(best_f1, 4),
                "acc_prior": round(float(accuracy_score(y_dev, predw)), 4),
                "w": np.round(w, 3).tolist(),
            })
            print(f"{variant:10s} C={C:<6} cw={str(cw):6s} | "
                  f"mf1={best_f1:.4f} acc={accuracy_score(y_dev, predw):.4f} | "
                  f"w={np.round(w, 2).tolist()}")

search_df = pd.DataFrame(rows).sort_values("mf1_prior", ascending=False).reset_index(drop=True)
print("\nTop 10 cấu hình theo macro-F1:")
print(search_df.head(10).drop(columns=["w"]).to_string(index=False))

BEST = search_df.iloc[0]
BEST_VARIANT = BEST["variant"]
BEST_C = float(BEST["C"])
BEST_CW = None if BEST["class_weight"] == "None" else BEST["class_weight"]
print(f"\nCHỌN: variant={BEST_VARIANT}, C={BEST_C}, class_weight={BEST_CW}, "
      f"macro-F1={BEST['mf1_prior']:.4f}")

default    C=0.003  cw=None   | mf1=0.7764 acc=0.9119 | w=[1.0, 1.5, 1.5, 1.1]
default    C=0.01   cw=None   | mf1=0.7764 acc=0.9119 | w=[0.9, 1.1, 1.25, 1.0]
default    C=0.03   cw=None   | mf1=0.7813 acc=0.9113 | w=[0.8, 1.0, 1.1, 1.1]
default    C=0.1    cw=None   | mf1=0.7802 acc=0.9119 | w=[0.8, 1.0, 1.0, 1.0]
default    C=0.3    cw=None   | mf1=0.7802 acc=0.9119 | w=[0.8, 1.1, 1.0, 1.0]
default    C=1.0    cw=None   | mf1=0.7794 acc=0.9113 | w=[0.8, 1.1, 1.0, 1.0]
default    C=10.0   cw=None   | mf1=0.7824 acc=0.9094 | w=[1.5, 1.1, 2.0, 1.0]
balanced   C=0.003  cw=None   | mf1=0.7719 acc=0.9063 | w=[1.1, 2.5, 2.0, 1.1]
balanced   C=0.01   cw=None   | mf1=0.7748 acc=0.9119 | w=[0.9, 2.5, 1.1, 1.1]
balanced   C=0.03   cw=None   | mf1=0.7716 acc=0.9113 | w=[0.9, 2.5, 1.0, 1.1]
balanced   C=0.1    cw=None   | mf1=0.7736 acc=0.9063 | w=[1.1, 2.5, 2.0, 1.0]
balanced   C=0.3    cw=None   | mf1=0.7725 acc=0.9063 | w=[1.1, 2.5, 2.0, 1.1]
balanced   C=1.0    cw=None   | mf1=0.7725 acc=0.90

## 6. Ablation trên variant tốt nhất

Bỏ lần lượt từng model, đo macro-F1 trên dev.

In [6]:
# ============================================================
# 6. ABLATION
# ============================================================
def fit_variant(variant, models=None, C=BEST_C, cw=BEST_CW):
    d = DATA[variant]
    old_models = d["models"]
    if models is not None:
        d["models"] = models
    Xtr = build_meta(variant, "train")
    Xdv = build_meta(variant, "dev")
    d["models"] = old_models
    meta = LogisticRegression(C=C, class_weight=cw, max_iter=3000,
                              solver="lbfgs", random_state=SEED)
    meta.fit(Xtr, y_train)
    p = meta.predict_proba(Xdv)
    w, score = tune_prior(p, y_dev)
    pred = (p * w).argmax(axis=1)
    return meta, w, pred, score


base_models = DATA[BEST_VARIANT]["models"]
ablation = []
for drop in base_models:
    subset = [m for m in base_models if m != drop]
    _, _, pred, score = fit_variant(BEST_VARIANT, models=subset)
    ablation.append({
        "dropped": drop,
        "macro_f1": round(score, 4),
        "accuracy": round(float(accuracy_score(y_dev, pred)), 4),
    })
    print(f"bỏ {drop:14s} → mf1={score:.4f}  acc={accuracy_score(y_dev, pred):.4f}")

print(f"\nĐầy đủ: mf1={BEST['mf1_prior']:.4f}, acc={BEST['acc_prior']:.4f}")
print(pd.DataFrame(ablation).sort_values("macro_f1", ascending=False).to_string(index=False))

bỏ lightgbm       → mf1=0.7733  acc=0.9101
bỏ linsvc         → mf1=0.7802  acc=0.9113
bỏ logreg         → mf1=0.7698  acc=0.9126
bỏ complement_nb  → mf1=0.7795  acc=0.9113
bỏ sgd            → mf1=0.7817  acc=0.9119
bỏ phobert        → mf1=0.7644  acc=0.8868

Đầy đủ: mf1=0.7830, acc=0.9132
      dropped  macro_f1  accuracy
          sgd    0.7817    0.9119
       linsvc    0.7802    0.9113
complement_nb    0.7795    0.9113
     lightgbm    0.7733    0.9101
       logreg    0.7698    0.9126
      phobert    0.7644    0.8868


## 7. Meta cuối & submission

Fit lại meta tốt nhất trên OOF train, áp prior, dự đoán test và ghi file nộp.

In [7]:
# ============================================================
# 7. FINAL & SUBMISSION
# ============================================================
Xtr = build_meta(BEST_VARIANT, "train")
Xdv = build_meta(BEST_VARIANT, "dev")
Xte = build_meta(BEST_VARIANT, "test")

meta_final = LogisticRegression(
    C=BEST_C, class_weight=BEST_CW, max_iter=3000, solver="lbfgs", random_state=SEED
)
meta_final.fit(Xtr, y_train)

p_dev = meta_final.predict_proba(Xdv)
p_test = meta_final.predict_proba(Xte)
w_opt, mf1_dev = tune_prior(p_dev, y_dev)

pred_dev = (p_dev * w_opt).argmax(axis=1)
pred_test = (p_test * w_opt).argmax(axis=1)

print("Variant:", BEST_VARIANT, "| models:", DATA[BEST_VARIANT]["models"])
print("Prior:", np.round(w_opt, 3).tolist())
print("Dev accuracy:", round(float(accuracy_score(y_dev, pred_dev)), 4))
print("Dev macro-F1:", round(macro_f1(y_dev, pred_dev), 4))
print("\nPhân bố nhãn dự đoán test:")
print(pd.Series(pred_test).value_counts().sort_index().to_string())

ids = pd.read_csv(ARTIFACT_DIR / "ids" / "test_ids.csv")["ID"].tolist()
assert len(ids) == len(pred_test) == 3974

submission = pd.DataFrame({"ID": ids, "spam_label": pred_test.astype(int)})
sample = pd.read_csv(PROJECT_ROOT / "Data" / "ds102_samplesubmission.csv", encoding="utf-8-sig")
assert list(submission.columns) == list(sample.columns)
assert submission["ID"].tolist() == sample["ID"].tolist()
assert submission["spam_label"].isin([0, 1, 2, 3]).all()

path = SUBMISSIONS_DIR / "submission_stacking.csv"
submission.to_csv(path, index=False, encoding="utf-8")
print("\nĐã ghi:", path, "| số dòng:", len(submission))

print("\nConfusion matrix dev:")
print(pd.DataFrame(
    confusion_matrix(y_dev, pred_dev, labels=[0, 1, 2, 3]),
    index=[f"true_{i}" for i in range(4)],
    columns=[f"pred_{i}" for i in range(4)],
).to_string())
print("\n", classification_report(y_dev, pred_dev, digits=3, zero_division=0))

joblib.dump({
    "model": meta_final,
    "variant": BEST_VARIANT,
    "base_models": DATA[BEST_VARIANT]["models"],
    "use_log_probs": USE_LOG_PROBS,
    "use_metadata": USE_METADATA,
    "meta_cols": META_COLS,
    "meta_scaler": meta_scaler,
    "prior_w": w_opt,
    "best_C": BEST_C,
    "class_weight": BEST_CW,
    "objective": "macro_f1",
    "dev_macro_f1": round(float(macro_f1(y_dev, pred_dev)), 4),
    "dev_accuracy": round(float(accuracy_score(y_dev, pred_dev)), 4),
}, META_MODEL_DIR / "meta_learner.joblib", compress=3)
print("Đã lưu meta model:", META_MODEL_DIR / "meta_learner.joblib")

Variant: sqrt | models: ['lightgbm', 'linsvc', 'logreg', 'complement_nb', 'sgd', 'phobert']
Prior: [1.5, 1.25, 2.5, 0.9]
Dev accuracy: 0.9132
Dev macro-F1: 0.783

Phân bố nhãn dự đoán test:
0    2957
1      38
2     334
3     645

Đã ghi: D:\DS102_NguyenKhacHuy_24520679\Final project\submissions\submission_stacking.csv | số dòng: 3974

Confusion matrix dev:
        pred_0  pred_1  pred_2  pred_3
true_0    1127       1      43      12
true_1       6       9       0       2
true_2      38       0      80       2
true_3      25       1       8     236

               precision    recall  f1-score   support

           0      0.942     0.953     0.947      1183
           1      0.818     0.529     0.643        17
           2      0.611     0.667     0.637       120
           3      0.937     0.874     0.904       270

    accuracy                          0.913      1590
   macro avg      0.827     0.756     0.783      1590
weighted avg      0.915     0.913     0.913      1590

Đã lưu m

## 8. Ghi chú

- `default` dùng dev/test từ model fit toàn train; `balanced`/`sqrt` dùng dev/test trung bình 5 fold — mỗi variant nhất quán nội bộ.
- Nếu variant có class weight thắng, đó là bằng chứng tầng 1 cần weight; nếu `default` vẫn thắng, giữ nguyên.
- Grid search và prior đều chọn trên dev → lạc quan nhẹ; ưu tiên xem chênh lệch tương đối.
- File nộp: `submissions/submission_stacking.csv`.